# 04 — Financial Inclusion EDA & Demographic Gap (Integrated with Notebook 03)

**Owner:** Person 1 (`rrrafaa`) | **Data:** World Bank Global Findex Indonesia survey 2024  
**Prerequisite:** Notebook **03** must have completed with **ALL INTEGRATION CHECKS PASSED** and its four required output files must be available privately.

**NEW DATA FLOW:** `03_cleaning_integration.ipynb` → validated private respondent dataset + canonical weighted KPI aggregates + QA + manifest → **this Notebook 04** → analysis charts + review-ready Person 1 aggregates → Notebook 06 → Looker Studio.

**Important:** This notebook **does not read the raw ZIP** or repeat the demographic cleaning from scratch. Visualizations use the **canonical aggregate KPI table produced by Notebook 03**. For an independent check, it recalculates rates against the **private integrated respondent dataset** and requires exact agreement (within CSV export rounding). Do not share or commit respondent-level data.

**Run on:** Google Colab (recommended) or local Jupyter. The notebook starts with zero saved execution outputs: run every cell and inspect the results before saving an executed copy to GitHub.

## 1. Load the PRIVATE results of Notebook 03 (Google Drive or local)

Notebook 03 produces these files in `data/processed/` and, if its final cell used `SAVE_TO_PRIVATE_DRIVE = True`, in **`MyDrive/Financial_Inclusion_Private/processed/`**:

- `integrated_respondents_private.csv` (**private** person-level file; never upload to public GitHub)
- `dashboard_kpi_aggregates.csv` (validated KPI for dashboard)
- `integration_quality_checks.csv` (all rows should be `PASS`)
- `integration_manifest.json` (original source checksum and shape)

**Colab:** sign in with the **same Google account** that ran Notebook 03. The cell mounts Google Drive if necessary.  
**Local:** put these four files in your locally cloned repository's `data/processed/` folder.  
**Other folder:** set `CUSTOM_PROCESSED_DIR` to its absolute path in the code cell.

If files are not found in either location, Colab offers an upload dialog for these **four processed outputs**. Do not upload the original ZIP to Notebook 04.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_NAME = "Indonesia-Financial-Inclusion-Digital-Finance-Analytics"
REPO_URL = f"https://github.com/rrrafaa/{REPO_NAME}.git"
IN_COLAB = "google.colab" in sys.modules

# Optional: set an absolute path to a DIFFERENT private output directory.
CUSTOM_PROCESSED_DIR = None
NEEDED = (
    "integrated_respondents_private.csv",
    "dashboard_kpi_aggregates.csv",
    "integration_quality_checks.csv",
    "integration_manifest.json",
)
EXPECTED_SOURCE_SHA = "fc50f0a843e289a3b9784ac81dce1018a8195388608575e6c8315c195073a64c"
EXPECTED_SOURCE_CSV_SHA = "77a50e351c16946ad107c08494efe9e8972ba628168fa385b771695971710093"

def locate_repo():
    options = [Path.cwd(), *Path.cwd().parents]
    if IN_COLAB:
        options.append(Path("/content") / REPO_NAME)
    for folder in options:
        if (folder / "data" / "raw" / "README.md").is_file():
            return folder
    if IN_COLAB:
        folder = Path("/content") / REPO_NAME
        if not folder.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(folder)], check=True)
        if (folder / "data" / "raw" / "README.md").is_file():
            return folder
    raise FileNotFoundError("Clone the GitHub repository and run Jupyter from within its folder.")

ROOT = locate_repo()
LOCAL_PROCESSED = ROOT / "data" / "processed"
LOCAL_PROCESSED.mkdir(parents=True, exist_ok=True)
PRIVATE_DRIVE = Path("/content/drive/MyDrive/Financial_Inclusion_Private/processed")

def complete(folder):
    return folder.is_dir() and all((folder / filename).is_file() for filename in NEEDED)

choices = []
if CUSTOM_PROCESSED_DIR:
    choices.append(Path(CUSTOM_PROCESSED_DIR).expanduser())
if IN_COLAB:
    choices.append(PRIVATE_DRIVE)
choices.append(LOCAL_PROCESSED)

DATA_DIR = next((folder for folder in choices if complete(folder)), None)
if DATA_DIR is None and IN_COLAB:
    from google.colab import drive
    print("Connecting the private Google Drive used by Notebook 03...")
    drive.mount("/content/drive")
    DATA_DIR = next((folder for folder in choices if complete(folder)), None)

if DATA_DIR is None and IN_COLAB:
    from google.colab import files
    print("Notebook 03 outputs were not found. Upload these FOUR private files:")
    print(*NEEDED, sep="\n - ")
    uploaded = files.upload()
    missing = set(NEEDED) - set(uploaded)
    if missing:
        raise FileNotFoundError(f"Missing output files: {sorted(missing)}")
    for filename in NEEDED:
        (LOCAL_PROCESSED / filename).write_bytes(uploaded[filename])
    DATA_DIR = LOCAL_PROCESSED

if DATA_DIR is None:
    raise FileNotFoundError(
        "Required Notebook 03 outputs are absent. Place all four files in data/processed/ "
        "or configure CUSTOM_PROCESSED_DIR."
    )

manifest = json.loads((DATA_DIR / "integration_manifest.json").read_text(encoding="utf-8"))
qa = pd.read_csv(DATA_DIR / "integration_quality_checks.csv")
assert manifest.get("source_zip_sha256") == EXPECTED_SOURCE_SHA, "Notebook 03 used a different ZIP."
assert manifest.get("source_csv_sha256") == EXPECTED_SOURCE_CSV_SHA, "Notebook 03 CSV checksum differs."
assert manifest.get("qa_passed") is True, "Notebook 03 QA flag was not PASS."
assert len(qa) >= 14 and qa["status"].eq("PASS").all(), "Notebook 03 has failed or missing QA checks."
assert manifest.get("raw_shape") == [1068, 199], "Unexpected source data shape."

d = pd.read_csv(DATA_DIR / "integrated_respondents_private.csv", low_memory=False)
canonical = pd.read_csv(DATA_DIR / "dashboard_kpi_aggregates.csv")
assert list(d.shape) == manifest["clean_shape"], "Integrated respondent file and manifest disagree."
assert len(d) == 1068 and set(d["year"].dropna().unique()) == {2024}
assert "wpid_random" not in d.columns, "Source respondent ID must not be used in this workflow."

print("Notebook 03 integration: VERIFIED")
print("Private input directory:", DATA_DIR)
print("Integrated observations:", len(d), "| Canonical KPI rows:", len(canonical))
print("Quality checks:", len(qa), "PASS")
print("No raw ZIP is required for this notebook.")

## 2. Use the same category definitions established by Notebook 03

All subgroup labels are already present in `integrated_respondents_private.csv` and must **not** be recoded independently. The single missing education response remains missing and is **excluded** from education comparisons. The source dataset is never displayed row-by-row.

In [ ]:
DIMENSION_ORDER = {
    "gender": ["Female", "Male"],
    "income_quintile": ["Q1 - lowest", "Q2", "Q3", "Q4", "Q5 - highest"],
    "education": ["Primary or less", "Secondary", "Tertiary or more"],
    "age_group": ["15-24", "25-34", "35-44", "45-54", "55+"],
    "residence": ["Rural", "Urban"],
    "internet_group": ["Non-internet user", "Internet user"],
}
INDICATORS = {
    "account": "Owns an account",
    "account_fin": "Financial institution account",
    "account_mob": "Mobile money account",
    "anydigpayment": "Digital payment use",
    "saved": "Saved money",
    "borrowed": "Borrowed money",
}
for column in set(INDICATORS) | {"internet_use", "wgt", "year", *DIMENSION_ORDER}:
    assert column in d.columns, f"Missing integrated column: {column}"
for indicator in set(INDICATORS) | {"internet_use"}:
    assert d[indicator].dropna().isin([0, 1]).all(), f"Invalid indicator code: {indicator}"
assert d["wgt"].notna().all() and d["wgt"].gt(0).all()
assert d["education"].isna().sum() == 1
for dimension, labels in DIMENSION_ORDER.items():
    unexpected = set(d[dimension].dropna().unique()) - set(labels)
    assert not unexpected, f"Unmapped labels in {dimension}: {unexpected}"
assert not d["age_group"].isna().any(), "Age groups must be complete."

expected_cols = {
    "metric", "metric_label", "dimension", "category", "sort_order",
    "numerator_wgt", "denominator_wgt", "rate_pct", "n_valid", "n_yes"
}
assert expected_cols <= set(canonical.columns)
assert canonical[["metric", "dimension", "category"]].notna().all().all()
assert canonical["n_valid"].ge(30).all(), "Published aggregates contain small cells."
assert canonical["rate_pct"].between(0, 100).all()
assert not canonical.duplicated(["metric", "dimension", "category"]).any()
assert set(canonical["metric"]) == set(INDICATORS) | {"internet_use"}
assert len(canonical) == 140, "Notebook 03 KPI table shape has changed; review before proceeding."

print("Canonical categories validated; education missingness retained.")
print("No respondent-level preview will be displayed.")

## 3. Recalculate and cross-check every KPI with Notebook 03

Each row of the 140-row canonical table is checked against an **independent computation** on the private integrated respondent data:

\[
\text{Weighted rate (\%)}=100\times \frac{\sum_i w_i\,\mathbb{1}(y_i=1)}{\sum_i w_i}\quad\text{over valid answers only.}
\]

**Important:** We compare unrounded numerators, denominators, rates and sample sizes. The CSV from Notebook 03 is rounded to 8 decimal places, so the comparison allows tiny floating-point export tolerances. **Stop if this cell fails**; do not use the resulting charts for the dashboard.

In [ ]:
def weighted_components(frame, indicator):
    valid = frame.loc[frame[indicator].isin([0, 1]) & frame["wgt"].gt(0)]
    if valid.empty:
        raise ValueError(f"No valid observations for {indicator}")
    numerator = float((valid[indicator] * valid["wgt"]).sum())
    denominator = float(valid["wgt"].sum())
    return {
        "numerator_wgt": numerator,
        "denominator_wgt": denominator,
        "rate_pct": 100.0 * numerator / denominator,
        "n_valid": int(len(valid)),
        "n_yes": int(valid[indicator].eq(1).sum())
    }

checked = []
for row in canonical.itertuples(index=False):
    if row.dimension == "national":
        frame = d
    else:
        if row.dimension not in DIMENSION_ORDER:
            raise ValueError(f"Unexpected aggregation dimension: {row.dimension}")
        frame = d.loc[d[row.dimension].eq(row.category)]
    expected = weighted_components(frame, row.metric)
    ok = (
        int(row.n_valid) == expected["n_valid"]
        and int(row.n_yes) == expected["n_yes"]
        and abs(float(row.numerator_wgt) - expected["numerator_wgt"]) <= 0.00001
        and abs(float(row.denominator_wgt) - expected["denominator_wgt"]) <= 0.00001
        and abs(float(row.rate_pct) - expected["rate_pct"]) <= 0.00001
    )
    checked.append({
        "metric": row.metric, "dimension": row.dimension, "category": row.category,
        "status": "PASS" if ok else "FAIL",
        "recomputed_rate_pct": expected["rate_pct"],
        "canonical_rate_pct": float(row.rate_pct),
        "n_valid": expected["n_valid"]
    })

validation = pd.DataFrame(checked)
if not validation.status.eq("PASS").all():
    display(validation.loc[validation.status.eq("FAIL")])
    raise AssertionError("Notebook 04 and Notebook 03 do not match. STOP dashboard work.")
assert len(validation) == 140
print("ALL 140 KPI CROSS-CHECKS PASSED.")
print("Source: Notebook 03 integrated respondent data + canonical weighted aggregates")

def by_group(dimension, indicator="account"):
    """Get the approved, sorted Notebook 03 weighted KPI table (not an average of group rates)."""
    if dimension not in DIMENSION_ORDER:
        raise ValueError(f"Unknown demographic dimension: {dimension}")
    part = canonical.loc[
        canonical["metric"].eq(indicator) & canonical["dimension"].eq(dimension)
    ].copy().sort_values("sort_order")
    return part.rename(columns={
        "category": "group",
        "denominator_wgt": "weight_total"
    }).assign(small_n_warning=lambda x: x.n_valid < 30)

overview = (
    canonical.loc[
        canonical["dimension"].eq("national")
        & canonical["metric"].isin(INDICATORS)
    ].copy()
    .assign(label=lambda x: x["metric"].map(INDICATORS))
)
overview = overview[["metric", "label", "rate_pct", "n_valid",
                     "numerator_wgt", "denominator_wgt"]]
overview["sort_order"] = overview["metric"].map({k: i for i, k in enumerate(INDICATORS)})
overview = overview.sort_values("sort_order").drop(columns="sort_order").reset_index(drop=True)
display(overview.round(3))
assert 50 < overview.loc[overview.metric.eq("account"), "rate_pct"].iloc[0] < 65

## 4. National financial inclusion overview

Use the **canonical Notebook 03 weighted rates** for the national KPIs, not respondent-level previews or a mean of subgroup percentages.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
tmp = overview.sort_values("rate_pct")
ax.barh(tmp["label"], tmp["rate_pct"])
ax.set_xlim(0, 100)
ax.set_xlabel("Survey-weighted percentage (%)")
ax.set_title("Indonesia 2024 — Financial inclusion overview (integrated data)")
for i, v in enumerate(tmp["rate_pct"]):
    ax.text(min(v + 1, 97), i, f"{v:.1f}%", va="center")
plt.tight_layout()
plt.show()

## 5. Account ownership by income quintile

Show Q1–Q5 in the agreed order. Compare **Q5 minus Q1** and separately distinguish the observed maximum–minimum difference (which may involve Q3). Report gaps in **percentage points**, not relative percent.

In [ ]:
income = by_group("income_quintile", "account")
display(income[["group", "rate_pct", "n_valid", "small_n_warning"]].round(3))
fig, ax = plt.subplots(figsize=(9, 4.3))
ax.bar(income["group"], income["rate_pct"])
ax.set(ylim=(0, 100), ylabel="Weighted account ownership (%)",
       title="Account ownership by income quintile (Notebook 03 source)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
q1 = float(income.loc[income.group.eq("Q1 - lowest"), "rate_pct"].iloc[0])
q5 = float(income.loc[income.group.eq("Q5 - highest"), "rate_pct"].iloc[0])
print(f"Income Q5 − Q1 gap: {q5 - q1:.3f} percentage points")
print(f"Observed max − min across income quintiles: {income.rate_pct.max() - income.rate_pct.min():.3f} pp")
assert q5 > q1

## 6. Account ownership by gender, education, age group and residence

The one respondent with unknown education is **not** a plotted category. All displayed groups come from the **Notebook 03 approved aggregations**, with at least 30 valid respondents (small groups require additional caution even when above 30).

In [ ]:
dimensions = ["gender", "education", "age_group", "residence"]
tables = []
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, dim in zip(axes.flat, dimensions):
    t = by_group(dim, "account")
    tables.append(t)
    ax.barh(t["group"], t["rate_pct"])
    ax.set_xlim(0, 100)
    ax.set_title("Account ownership — " + dim.replace("_", " ").title())
    ax.set_xlabel("Survey-weighted percentage (%)")
    for i, v in enumerate(t["rate_pct"]):
        ax.text(min(v + 1, 97), i, f"{v:.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()
segments = pd.concat([income, *tables], ignore_index=True)
assert not segments["group"].isna().any()
display(segments[["dimension", "group", "rate_pct", "n_valid", "small_n_warning"]].round(3))
print("Small cells (<30 unweighted valid respondents):", int(segments.small_n_warning.sum()))
print("Note: tertiary education has only 41 unweighted respondents; treat estimates cautiously.")

## 7. Compare different financial inclusion indicators by demographic group

The original indicators **may overlap**, so never stack them to 100%. Values below come from the canonical Notebook 03 aggregates. The chart compares the same indicators across genders for descriptive insight.

In [ ]:
cross = pd.concat(
    [by_group(dim, indicator) for dim in ["gender", "income_quintile", "residence"]
     for indicator in INDICATORS],
    ignore_index=True
)
display(cross[["dimension", "group", "metric", "rate_pct", "n_valid"]].head(18).round(3))
pivot = (
    cross.loc[cross.dimension.eq("gender")]
    .pivot(index="metric", columns="group", values="rate_pct")
    .reindex(index=list(INDICATORS))
)
pivot.index = [INDICATORS[k] for k in pivot.index]
display(pivot.round(2))
ax = pivot.plot(kind="bar", figsize=(11, 5))
ax.set_ylim(0, 100)
ax.set_ylabel("Survey-weighted percentage (%)")
ax.set_title("Financial inclusion indicators by gender (Notebook 03 source)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 8. Financial inclusion gap table and final consistency tests

For each demographic dimension, show the **largest observed descriptive gap** among approved groups. This differs from a policy-specific Q5–Q1 comparison. These are **not** significance tests.

In [ ]:
gap_rows = []
for dim in dimensions + ["income_quintile"]:
    t = by_group(dim, "account")
    valid = t.loc[~t["small_n_warning"] & t["rate_pct"].notna()]
    if len(valid) < 2:
        continue
    hi = valid.loc[valid.rate_pct.idxmax()]
    lo = valid.loc[valid.rate_pct.idxmin()]
    gap_rows.append({
        "dimension": dim, "highest_group": hi.group, "lowest_group": lo.group,
        "max_rate_pct": float(hi.rate_pct),
        "min_rate_pct": float(lo.rate_pct),
        "gap_pp": float(hi.rate_pct - lo.rate_pct)
    })
gaps = pd.DataFrame(gap_rows).sort_values("gap_pp", ascending=False).reset_index(drop=True)
display(gaps.round(3))
national_source = float(overview.loc[overview.metric.eq("account"), "rate_pct"].iloc[0])
national_recalc = weighted_components(d, "account")["rate_pct"]
assert abs(national_source - national_recalc) < 1e-5
assert not segments.rate_pct.dropna().gt(100).any()
print(f"National weighted account ownership: {national_source:.3f}% (matches Notebook 03)")
print("Verified survey respondents:", len(d), "| Year:", d.year.unique().tolist())
print("KPI cross-check status: ALL PASS")

## 9. Save approved Person 1 aggregates for Notebook 06

Four **aggregate-only** outputs are generated from the canonical Notebook 03 figures. If Notebook 03 results were loaded from **Google Drive**, these four files are saved to the same **private** Drive folder; otherwise, they are saved to local `data/processed/`.

The individual-level dataset remains private. **Do not push CSV data to the public repository by force**. Before uploading aggregates to Google Sheets / Looker Studio, both analysts should confirm the final metrics and any small-cell disclosure rules.

In [ ]:
# Save results alongside validated Notebook 03 outputs; no public GitHub upload happens here.
out = DATA_DIR if DATA_DIR == PRIVATE_DRIVE else LOCAL_PROCESSED
out.mkdir(parents=True, exist_ok=True)
exports = {
    "person1_overview_kpis.csv": overview,
    "person1_account_segments.csv": segments,
    "person1_cross_metrics.csv": cross,
    "person1_gap_summary.csv": gaps,
    "person1_notebook03_validation.csv": validation
}
for filename, frame in exports.items():
    frame.to_csv(out / filename, index=False, float_format="%.8f")
print("Saved five approved analysis/validation tables in:", out)
for filename in exports:
    print(" -", filename)
print("Do not commit respondent-level data or source ZIP files.")

## 10. Interpretation of EDA Results (Person 1)

**Status:** Integrated-data EDA. Notebook 04 now uses the **private cleaned data and the canonical 140-row KPI aggregates produced by Notebook 03**. Its 140-row consistency validation must all PASS before interpreting these findings. The numeric examples below are the prior survey-weighted benchmarks and should be reconfirmed after this notebook is rerun.

### 10.1 National financial inclusion — the big picture

| Indicator | Survey-weighted share |
|---|---:|
| Owns any account (`account`) | **56.33%** |
| Financial institution account (`account_fin`) | 52.39% |
| Mobile money account (`account_mob`) | 22.01% |
| Used digital payment (`anydigpayment`) | 42.69% |
| Saved money (`saved`) | 52.99% |
| Borrowed money (`borrowed`) | 47.64% |

**Interpretation:** Account ownership is above 50%, but the survey still points to an important inclusion gap. Use of digital payments (42.69%) is below overall account ownership (56.33%). These are **different, potentially overlapping indicators**; do not treat their difference as a measured conversion funnel or add the rates together.

### 10.2 Income gradient — a large gap, but not perfectly monotonic

| Income quintile | Weighted account ownership |
|---|---:|
| Q1, lowest income | 46.425% |
| Q2 | 50.406% |
| Q3 | 46.203% |
| Q4 | 60.590% |
| Q5, highest income | 78.002% |

The **Q5 minus Q1 gap is 31.577 percentage points** (about **31.58 pp**). The *maximum minus minimum among all quintiles* is **31.799 pp**, comparing Q5 with Q3; this is why the Notebook 04 gap table lists Q3 rather than Q1 as its minimum. The pattern is **not perfectly monotonic**, because Q3 is marginally below Q1. Avoid saying that the ownership rate necessarily rises with every income quintile.

**Business/policy question to investigate:** Could affordability, onboarding and access barriers partly explain the lower account ownership among lower-income groups? This analysis cannot establish causal mechanisms.

### 10.3 Education — largest observed descriptive gap

| Education group | Weighted ownership | Unweighted n |
|---|---:|---:|
| Primary or less | 37.638% | 380 |
| Secondary | 66.268% | 646 |
| Tertiary or more | 85.607% | 41 |

The difference between **tertiary or more** and **primary or less** is **47.969 pp** (about **47.97 pp**), the largest *descriptive* gap among the segment dimensions examined. The tertiary group has only **41 respondents**, so do **not** claim the gap is statistically significant or causally attributable to education.

**Data-quality note:** One respondent has missing `educ`. The original Colab output displayed that missing segment with `n=1` and a 0% rate. This category **must be excluded from substantive comparisons and public charts**, not interpreted as an education category. Notebook 03 excludes missing education from published subgroup aggregates. The updated plotting cell in Notebook 04 uses those aggregates and must be rerun to regenerate its chart.

### 10.4 Age — a pronounced descriptive gradient

| Age group | Weighted ownership |
|---|---:|
| 15–24 | 61.993% |
| 25–34 | **70.135%** |
| 35–44 | 61.054% |
| 45–54 | 45.226% |
| 55+ | **39.519%** |

The difference between the highest age group rate (25–34) and the lowest (55+) is **30.616 pp** (about **30.62 pp**). This suggests an age-related inclusion disparity worth exploring. It does not show whether age itself causes the difference.

### 10.5 Gender and rural/urban residence — comparatively small differences

| Segment | Weighted ownership |
|---|---:|
| Female | 58.031% |
| Male | 54.639% |
| Urban | 56.666% |
| Rural | 56.030% |

The female–male difference is **3.392 pp**; the urban–rural difference is **0.636 pp**. Both are much smaller *descriptively* than the education, income and age gaps. **Neither was tested for statistical significance.** Do not generalize to specific Indonesian provinces: this file contains no province identifier.

Other gender comparisons in the output show **mobile money** account ownership at **24.131% for men vs 19.878% for women**, while saving is **56.463% for women vs 49.555% for men**. Inclusion dimensions can have different patterns; avoid an oversimplified claim that one gender always leads.

### 10.6 Suggested dashboard narrative (Person 1 — Pages 1 and 2)

- **Page 1: Financial Inclusion Overview.** Present six weighted KPI cards, a comparative bar chart of indicators, survey year (2024), sample size (n=1,068), and a short methodology note on weights.
- **Page 2: Inclusion Gap & Demographics.** Compare ownership by income Q1–Q5, education, age group, gender and rural/urban residence. Show **percentage-point** gaps instead of ambiguous percent changes, and add unweighted subgroup n as a contextual quality measure.
- **Headline:** "Income, education and age disparities are much more pronounced than the gender or urban/rural differences in this 2024 descriptive sample."
- **Candidate interventions to explore, not proven causal recommendations:** affordability and accessible services for lower-income groups; inclusive account-opening support; age-friendly financial onboarding; and evidence gathering on financial literacy or product usability.

### 10.7 Methodology and limitations

1. Shares use Notebook 03's canonical aggregated weighted numerator and denominator, checked independently against its private integrated respondent file: `sum(wgt × binary_indicator) / sum(wgt)` over valid responses. Percentage-point gaps are differences between weighted shares.
2. This is a **single-year cross-sectional survey (2024)**. It cannot establish causal effects or trends over multiple years.
3. Small groups require caution: the tertiary education subgroup has `n=41` and the missing education subgroup has `n=1` (excluded). The rule `n<30` is a **simple flag**, not a statistical uncertainty estimate.
4. No survey-design-based confidence intervals or hypothesis tests were computed. In particular, small gender and residence differences should **not** be described as statistically meaningful.
5. Notebook 03 validates binary coding, the official source checksum and survey routing. Interpret eligible response universes and time windows against Notebook 02 and the World Bank codebook; never interpret skipped values as zero.
6. Keep the raw ZIP and respondent-level transformed data out of public GitHub. Notebook 04 has no respondent preview and intentionally stores no stale output from the earlier raw-data run.

### 10.8 Next steps / handoff checklist

- [x] Execute EDA on validated original microdata and inspect weighted results.
- [x] Document preliminary interpretation and main descriptive gaps.
- [x] Notebook 02 is in GitHub; the previously incorrect 2024 barrier labels have been corrected in its source code (Person 2 should rerun their affected outputs).
- [x] Notebook 03 is available and has validated the integration; Notebook 04 code now **consumes its outputs**.
- [ ] **Run this updated Notebook 04 in Colab**, confirm all 140 KPI cross-checks PASS and inspect the regenerated plots.
- [ ] Review small samples, benchmark national indicators to official Findex definitions and check all denominators.
- [ ] Export the five Person 1 aggregate and validation CSVs privately; confirm they are present in your Drive or local `data/processed/`.
- [ ] Have Person 2 complete Notebook 05, then use Notebook 06 to approve/export only aggregated tables to Google Sheets or Looker Studio.
- [ ] Build Page 1 and Page 2; cross-review calculations, filters, chart labels and storyline.
- [ ] Add reviewed dashboard screenshots and the Looker Studio link to `dashboard/README.md`.
